# 03 — System Comparison: Naive / Static Hybrid / Adaptive

Query analyzer (Table 20), three-system retrieval comparison with Wilcoxon+Holm
(Table 21), and by-complexity analysis (Table 22). Bootstrap CIs included.


In [ ]:
# Configuration is centralised in src/config.py (single source of truth).
# If running outside the package, the constants below mirror that file.
import sys, os
sys.path.append(os.path.abspath(".."))
try:
    from src.config import *
    print("Loaded config from src.config")
except Exception as e:
    print("src.config not importable here; using inline constants.", e)
    RANDOM_SEED = 42
    RRF_KAPPA = 60
    DENSE_BATCH_SIZE = 128
    MAX_SEQ_LENGTH = 512
    NAIVE_K, STATIC_K = 5, 10
    ADAPTIVE_K = {"simple":5,"moderate":10,"complex":15}


## Query analyzer (Table 20)

In [ ]:
#@title 7.1 Rule-based query analyzer using only question text
DEFINITION_MARKERS = [
    'дегеніміз не', 'анықтамасы', 'не болып табылады', 'кім болып табылады',
    'қандай ұғым', 'нені білдіреді', 'түсінігі'
]
DEADLINE_MARKERS = [
    'мерзім', 'неше күн', 'қанша күн', 'қанша уақыт', 'неше ай', 'қанша ай',
    'неше жыл', 'қанша жыл', 'күн ішінде', 'ай ішінде', 'жыл ішінде', 'дейін', 'кейін', 'бастап'
]
COMPLEX_MARKERS = [
    'қандай жағдайларда', 'қандай шарттар', 'қандай негіздер', 'қандай тәртіппен',
    'тәртібі', 'негіздері', 'ерекшеліктері', 'қоспағанда', 'егер', 'болған жағдайда',
    'бірнеше', 'қалай жүзеге асырылады', 'қалай белгіленеді'
]
UNANSWERABLE_RISK_MARKERS = [
    'болашақта', 'келесі жылы', '2026', '2027', 'болжам', 'қандай болады',
    'саяси', 'пікір', 'кеңес бер', 'адвокатсыз', 'сотта жеңемін бе'
]


def contains_any(text, markers):
    return any(m in text for m in markers)


def analyze_question(question: str):
    q = str(question).lower().strip()
    unans_risk = contains_any(q, UNANSWERABLE_RISK_MARKERS)

    if unans_risk:
        qtype = 'unanswerable_risk'
        complexity = 'complex'
        route = 'unanswerable_risk'
    elif contains_any(q, DEFINITION_MARKERS):
        qtype = 'definition'
        complexity = 'simple'
        route = 'definition_lookup'
    elif contains_any(q, DEADLINE_MARKERS):
        qtype = 'deadline'
        complexity = 'moderate'
        route = 'deadline_lookup'
    elif contains_any(q, COMPLEX_MARKERS):
        qtype = 'complex_legal'
        complexity = 'complex'
        route = 'complex_lookup'
    else:
        qtype = 'moderate_lookup'
        complexity = 'moderate'
        route = 'moderate_lookup'

    return {
        'pred_question_type': qtype,
        'pred_complexity': complexity,
        'pred_route': route,
        'pred_unanswerable_risk': int(unans_risk),
    }

qa_pred = pd.DataFrame([analyze_question(q) for q in bench_all['question']])
query_analyzer_df = pd.concat([bench_all[['row_id', 'question_id', 'question', 'answerability', 'complexity', 'question_type', 'legal_domain']], qa_pred], axis=1)
display(query_analyzer_df.head())
save_table(query_analyzer_df, 'detail05_query_analyzer_predictions')

In [ ]:
#@title 7.2 Post-hoc Query Analyzer evaluation against gold labels
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

qa_eval_rows = []
for gold_col, pred_col, task in [
    ('complexity', 'pred_complexity', 'Complexity prediction'),
    ('question_type', 'pred_question_type', 'Question type prediction'),
]:
    gold = query_analyzer_df[gold_col].astype(str)
    pred = query_analyzer_df[pred_col].astype(str)
    labels = sorted(set(gold) | set(pred))
    qa_eval_rows.append({
        'Task': task,
        'N': len(query_analyzer_df),
        'Accuracy': round(accuracy_score(gold, pred), 4),
        'MacroF1': round(f1_score(gold, pred, labels=labels, average='macro', zero_division=0), 4),
    })

# Unanswerable-risk detection: map answerability=unanswerable to positive class.
gold_unans = query_analyzer_df['answerability'].eq('unanswerable').astype(int)
pred_unans = query_analyzer_df['pred_unanswerable_risk'].astype(int)
qa_eval_rows.append({
    'Task': 'Unanswerable-risk detection',
    'N': len(query_analyzer_df),
    'Accuracy': round(accuracy_score(gold_unans, pred_unans), 4),
    'MacroF1': round(f1_score(gold_unans, pred_unans, average='macro', zero_division=0), 4),
})

qa_eval_df = pd.DataFrame(qa_eval_rows)
display(qa_eval_df)
save_table(qa_eval_df, 'table05_query_analyzer_posthoc_metrics')

route_dist = query_analyzer_df['pred_route'].value_counts().rename_axis('route').reset_index(name='count')
display(route_dist)
save_table(route_dist, 'table05_query_analyzer_route_distribution')

# Save crosstabs for error analysis.
ct_complexity = pd.crosstab(query_analyzer_df['complexity'], query_analyzer_df['pred_complexity'])
ct_qtype = pd.crosstab(query_analyzer_df['question_type'], query_analyzer_df['pred_question_type'])
ct_complexity.to_csv(OUTPUT_DIR / 'table05_crosstab_complexity.csv', encoding='utf-8-sig')
ct_qtype.to_csv(OUTPUT_DIR / 'table05_crosstab_question_type.csv', encoding='utf-8-sig')
print('Saved query analyzer crosstabs.')

## Three systems on 182 answerable (Table 21) + Wilcoxon signed-rank + Holm

In [ ]:
#@title 8.1 Three-system retrieval policies — Naive / Static Hybrid / Adaptive (dynamic top-k, no rerank)
# Methodology (paper Section 3.6): adaptivity = COMPLEXITY-CONDITIONED DYNAMIC TOP-K.
#   simple  -> k=5  (compact evidence)
#   moderate-> k=10 (balanced)
#   complex -> k=15 (broad multi-provision)
# Fusion = BM25 + BGE-M3 via RRF + dedup. NO reranker (rerank is a separate ablation, Section 4.5).
# Oracle routing is NOT used in the paper; only Naive / Static / Adaptive are compared.

NAIVE_K = 5          # Naive RAG: BM25 only, fixed k
STATIC_K = 10        # Static Hybrid: BM25+BGE-M3, fixed k
ADAPTIVE_DEPTH = {'simple': 5, 'moderate': 10, 'complex': 15}
ADAPTIVE_DEFAULT_K = 10

def naive_retrieve(question, k=NAIVE_K):
    # Minimal non-adaptive baseline: sparse BM25 only.
    return bm25_ret(bm25_text, question, k=k, tokenizer=tok)

def static_hybrid_retrieve(question, query_embedding, k=STATIC_K):
    # Fixed-policy hybrid: same k for every question.
    return rrf([bm25_ret(bm25_text, question, k=k, tokenizer=tok),
                dense_ret(faiss_index, query_embedding, k=k)], k=k)

def adaptive_retrieve(question, query_embedding, depth_k):
    # Complexity-conditioned hybrid: dynamic k, same fusion as static.
    return rrf([bm25_ret(bm25_text, question, k=depth_k, tokenizer=tok),
                dense_ret(faiss_index, query_embedding, k=depth_k)], k=depth_k)

def run_naive():
    return [naive_retrieve(row['question'])
            for _, row in tqdm(list(bench_ans.iterrows()), desc='Naive RAG (BM25 k=5)')]

def run_static():
    return [static_hybrid_retrieve(row['question'], query_embs_ans[i])
            for i, row in tqdm(list(bench_ans.iterrows()), desc='Static Hybrid (k=10)')]

def run_adaptive():
    out, routes = [], []
    for i, row in tqdm(list(bench_ans.iterrows()), desc='Adaptive (dynamic top-k)'):
        pred = analyze_question(row['question'])
        # Map predicted complexity -> retrieval depth (paper 3.6.3 / Table 9).
        depth_k = ADAPTIVE_DEPTH.get(str(pred.get('pred_complexity', '')).lower(), ADAPTIVE_DEFAULT_K)
        out.append(adaptive_retrieve(row['question'], query_embs_ans[i], depth_k))
        routes.append({**pred, 'depth_k': depth_k})
    return out, pd.DataFrame(routes)

print('Stage-5 retrieval policies ready: Naive / Static Hybrid / Adaptive (dynamic top-k, no rerank).')


In [ ]:
#@title 8.2 Run the three systems on 182 answerable questions
start = time.time(); naive_res = run_naive(); naive_lat = (time.time()-start)/len(bench_ans)*1000
start = time.time(); static_res = run_static(); static_lat = (time.time()-start)/len(bench_ans)*1000
start = time.time(); adaptive_res, adaptive_routes = run_adaptive(); adaptive_lat = (time.time()-start)/len(bench_ans)*1000

systems = [
    ('Naive RAG', naive_res, naive_lat),
    ('Static Hybrid Legal RAG', static_res, static_lat),
    ('Adaptive Legal RAG', adaptive_res, adaptive_lat),
]

sys_rows, sys_details = [], []
for name, results, latency in systems:
    metrics, details = eval_retrieval(results, bench_ans)
    row = {'System': name, 'N': len(bench_ans), 'Latency_ms': round(latency, 2)}
    row.update(metrics)
    sys_rows.append(row)
    sys_details.append(details.assign(System=name))

three_sys_df = pd.DataFrame(sys_rows)
display(three_sys_df)
save_table(three_sys_df, 'table_three_systems_overall')
save_table(pd.concat(sys_details, ignore_index=True), 'detail_three_systems')
save_table(adaptive_routes, 'detail_adaptive_routes')

# keep a dict for downstream (significance, complexity breakdown)
three_sys_details = {name: det for (name, _, _), det in zip(systems, sys_details)}
print('Done. Naive/Static/Adaptive evaluated on', len(bench_ans), 'answerable questions.')


In [ ]:
#@title 8.4 Significance testing — Wilcoxon signed-rank (paired) + Holm correction
from scipy.stats import wilcoxon
from itertools import combinations

SIG_METRICS = ['Hit@1', 'Recall@10', 'MRR@10', 'nDCG@10']
SYS_NAMES = ['Naive RAG', 'Static Hybrid Legal RAG', 'Adaptive Legal RAG']

# align per-question rows by row_id across systems
aligned = {name: three_sys_details[name].set_index('row_id') for name in SYS_NAMES}
common = set.intersection(*[set(d.index) for d in aligned.values()])
common = sorted(common)

sig_rows = []
for metric in SIG_METRICS:
    for a, b in combinations(SYS_NAMES, 2):
        va = aligned[a].loc[common, metric].astype(float).to_numpy()
        vb = aligned[b].loc[common, metric].astype(float).to_numpy()
        diff = va - vb
        if (diff == 0).all():
            stat, p = float('nan'), 1.0
        else:
            try:
                stat, p = wilcoxon(va, vb, zero_method='wilcox')
            except ValueError:
                stat, p = float('nan'), 1.0
        sig_rows.append({'Metric': metric, 'System A': a, 'System B': b,
                         'mean_A': round(float(va.mean()), 4),
                         'mean_B': round(float(vb.mean()), 4),
                         'p_raw': p})

sig_df = pd.DataFrame(sig_rows)
# Holm-Bonferroni within each metric family
sig_df['p_holm'] = float('nan')
for metric in SIG_METRICS:
    m = sig_df['Metric'] == metric
    order = sig_df[m].sort_values('p_raw').index
    n = len(order)
    prev = 0.0
    for rank, idx in enumerate(order):
        adj = min(1.0, (n - rank) * sig_df.loc[idx, 'p_raw'])
        adj = max(adj, prev)  # enforce monotonicity
        sig_df.loc[idx, 'p_holm'] = adj
        prev = adj
sig_df['sig_0.05'] = sig_df['p_holm'] < 0.05
sig_df['p_raw'] = sig_df['p_raw'].round(4)
sig_df['p_holm'] = sig_df['p_holm'].round(4)
display(sig_df)
save_table(sig_df, 'table_significance_wilcoxon')
print('Wilcoxon signed-rank (paired) on', len(common), 'common questions; Holm-corrected within each metric.')
print('sig_0.05=True means the paired difference is statistically significant after correction.')


## Retrieval by query complexity (Table 22)

In [ ]:
#@title 8.3 Table 21 — Retrieval performance by query complexity (Static vs Adaptive)
# Single-gold benchmark: R@1 == Recall@1, so only R@1 is shown.
_static = three_sys_details['Static Hybrid Legal RAG']
_adapt  = three_sys_details['Adaptive Legal RAG']
COMPLEXITY_ORDER = ['simple', 'moderate', 'complex']

t21 = []
for comp in COMPLEXITY_ORDER:
    for label, det in [('Static Hybrid Legal RAG', _static), ('Adaptive Legal RAG', _adapt)]:
        sub = det[det['complexity'].astype(str).str.lower() == comp]
        t21.append({
            'Complexity Level': comp.capitalize(), 'System': label, 'n': len(sub),
            'R@1':       round(float(sub['Hit@1'].mean()), 3) if len(sub) else None,
            'Recall@10': round(float(sub['Recall@10'].mean()), 3) if len(sub) else None,
            'MRR':       round(float(sub['MRR@10'].mean()), 3) if len(sub) else None,
            'nDCG@10':   round(float(sub['nDCG@10'].mean()), 3) if len(sub) else None,
        })
table21 = pd.DataFrame(t21)
display(table21)
save_table(table21, 'table21_retrieval_by_complexity')
print('Table 21 ready. n per complexity among 182 answerable (simple 42 / moderate 65 / complex 75).')


## Full-242 retrieval (for downstream generation/abstention)

In [ ]:
#@title 9.1 Run Static Hybrid and Adaptive retrieval for the full 242-row benchmark
# Uses the stage-5 functions (static_hybrid_retrieve, adaptive depth map), NOT the old
# hybrid_retrieve/ROUTE_CONFIG. Runs on all 242 (answerable + unanswerable) for abstention.

def static_hybrid_for_all():
    out = []
    for i, row in tqdm(list(bench_all.iterrows()), desc='Static Hybrid full benchmark'):
        out.append(static_hybrid_retrieve(row['question'], query_embs_all[i], k=STATIC_K))
    return out

def adaptive_for_all():
    out, routes = [], []
    for i, row in tqdm(list(bench_all.iterrows()), desc='Adaptive full benchmark'):
        pred = analyze_question(row['question'])
        depth_k = ADAPTIVE_DEPTH.get(str(pred.get('pred_complexity','')).lower(), ADAPTIVE_DEFAULT_K)
        out.append(adaptive_retrieve(row['question'], query_embs_all[i], depth_k))
        routes.append({**pred, 'depth_k': depth_k})
    return out, pd.DataFrame(routes)

static_all_res = static_hybrid_for_all()
adaptive_all_res, adaptive_routes_all = adaptive_for_all()

abst_base = bench_all[['row_id','question_id','question','answerability','complexity',
                       'question_type','legal_domain']].copy()
abst_base['static_confidence']   = [retrieval_confidence(r) for r in static_all_res]
abst_base['adaptive_confidence'] = [retrieval_confidence(r) for r in adaptive_all_res]
abst_base = pd.concat([abst_base.reset_index(drop=True),
                       adaptive_routes_all.add_prefix('adaptive_').reset_index(drop=True)], axis=1)
display(abst_base.head())
save_table(abst_base, 'detail08_full_benchmark_retrieval_confidence')
print(f'Done: {len(bench_all)} questions, confidence scores for Static and Adaptive saved.')

In [ ]:
#@title 9.2 Threshold sweep: row-level and unique-question-level metrics

def abstention_metrics_from_decisions(df, answer_col='pred_answer'):
    # pred_answer=True means system answers; pred_answer=False means abstains.
    ans = df['answerability'].eq('answerable')
    unans = df['answerability'].eq('unanswerable')
    pred_answer = df[answer_col].astype(bool)
    out = {
        'N': len(df),
        'AnswerableN': int(ans.sum()),
        'UnanswerableN': int(unans.sum()),
        'AbstentionAccuracy': round(float(((ans & pred_answer) | (unans & ~pred_answer)).mean()), 4),
        'CorrectAnswerRate': round(float((pred_answer[ans]).mean()), 4) if ans.any() else np.nan,
        'CorrectRefusalRate': round(float((~pred_answer[unans]).mean()), 4) if unans.any() else np.nan,
        'FalseAnswerRate': round(float((pred_answer[unans]).mean()), 4) if unans.any() else np.nan,
        'FalseAbstentionRate': round(float((~pred_answer[ans]).mean()), 4) if ans.any() else np.nan,
    }
    # Balanced decision score gives equal weight to answering answerable questions and refusing unanswerable questions.
    out['BalancedDecisionScore'] = round(float(np.nanmean([out['CorrectAnswerRate'], out['CorrectRefusalRate']])), 4)
    return out


def threshold_sweep(df, score_col, system_name):
    scores = df[score_col].fillna(0).astype(float).to_numpy()
    thresholds = np.unique(np.quantile(scores, np.linspace(0, 1, 101)))
    thresholds = np.unique(np.concatenate([[scores.min() - 1e-9], thresholds, [scores.max() + 1e-9]]))
    rows = []
    for th in thresholds:
        tmp = df.copy()
        tmp['pred_answer'] = tmp[score_col] >= th
        row = {'System': system_name, 'Threshold': float(th), 'Level': 'row'}
        row.update(abstention_metrics_from_decisions(tmp))
        rows.append(row)
        # Unique-question-level: duplicates are collapsed by question text. Confidence is averaged per question.
        uniq = tmp.groupby('question', as_index=False).agg({
            'answerability': 'first',
            score_col: 'mean',
            'pred_answer': 'first',
        })
        row = {'System': system_name, 'Threshold': float(th), 'Level': 'unique_question'}
        row.update(abstention_metrics_from_decisions(uniq))
        rows.append(row)
    return pd.DataFrame(rows)

sweep_static = threshold_sweep(abst_base, 'static_confidence', 'Static Hybrid RAG')
sweep_adaptive = threshold_sweep(abst_base, 'adaptive_confidence', 'Adaptive LegalRAG')
abst_sweep_df = pd.concat([sweep_static, sweep_adaptive], ignore_index=True)
save_table(abst_sweep_df, 'table08_abstention_threshold_sweep')

# Baseline that always answers.
always_answer = abst_base.copy()
always_answer['pred_answer'] = True
no_abst_row = {'System': 'No abstention: always answer', 'Threshold': np.nan, 'Level': 'row'}
no_abst_row.update(abstention_metrics_from_decisions(always_answer))

best_rows = []
for system in abst_sweep_df['System'].unique():
    for level in ['row', 'unique_question']:
        sub = abst_sweep_df[(abst_sweep_df['System'].eq(system)) & (abst_sweep_df['Level'].eq(level))].copy()
        best = sub.sort_values(['BalancedDecisionScore', 'AbstentionAccuracy'], ascending=False).iloc[0].to_dict()
        best['Selection'] = 'diagnostic_best_threshold_on_full_benchmark'
        best_rows.append(best)

abst_best_df = pd.concat([pd.DataFrame([no_abst_row]), pd.DataFrame(best_rows)], ignore_index=True)
display(abst_best_df)
save_table(abst_best_df, 'table08_abstention_summary')

print('Important reporting note: threshold-selected rows are diagnostic because the benchmark has no separate calibration split.')

## Bootstrap 95% CIs

In [ ]:
#@title 12b.1 Bootstrap 95% CI per system (resampling per-question scores)
import numpy as np
import pandas as pd

BOOTSTRAP_N = 1000
CI_LOW, CI_HIGH = 2.5, 97.5
_rng = np.random.default_rng(RANDOM_SEED)
CI_METRICS = ['Hit@1', 'Recall@5', 'Recall@10', 'MRR@10', 'nDCG@10']

def bootstrap_ci(values, n=BOOTSTRAP_N):
    """95% percentile bootstrap CI for the mean of per-question metric values."""
    v = np.asarray(pd.to_numeric(pd.Series(values), errors='coerce').dropna(), dtype=float)
    if len(v) == 0:
        return (np.nan, np.nan, np.nan)
    means = v[_rng.integers(0, len(v), size=(n, len(v)))].mean(axis=1)
    return float(v.mean()), float(np.percentile(means, CI_LOW)), float(np.percentile(means, CI_HIGH))

# Collect per-question detail frames produced earlier in the notebook.
# retrieval_details (cell 4.3) + sva_details (cell 8.2) cover all reported systems.
SYSTEM_DETAILS = {}
if 'retrieval_details' in dir():
    for sysname, det in retrieval_details.items():
        SYSTEM_DETAILS[sysname] = det
if 'sva_details' in dir():
    for det in sva_details:
        if 'System' in det.columns:
            SYSTEM_DETAILS[str(det['System'].iloc[0])] = det

ci_rows = []
for system, det in SYSTEM_DETAILS.items():
    for m in CI_METRICS:
        if m not in det.columns:
            continue
        mean, lo, hi = bootstrap_ci(det[m])
        ci_rows.append({
            'System': system, 'Metric': m, 'N': int(pd.to_numeric(det[m], errors='coerce').notna().sum()),
            'Mean': round(mean, 4), 'CI95_low': round(lo, 4), 'CI95_high': round(hi, 4),
            'CI_str': f'{mean:.3f} [{lo:.3f}, {hi:.3f}]',
        })

ci_df = pd.DataFrame(ci_rows)
display(ci_df)
save_table(ci_df, 'table11_bootstrap_ci_long')

if len(ci_df):
    ci_wide = ci_df.pivot(index='System', columns='Metric', values='CI_str').reset_index()
    display(ci_wide)
    save_table(ci_wide, 'table11_bootstrap_ci_wide')

print(f'Bootstrap CI: {BOOTSTRAP_N} resamples of per-question scores, 95% percentile interval.')
